# Technology Dimension: AI research and patenting intensity

`ai_publication_share` (AI publications / total publications) and
`ai_patent_share` (AI patents / total patents), both from ETO PARAT company
aggregates. The shares measure how strongly a firm's research and patenting
activity is oriented toward AI, independent of firm size.

Missing is marked, not dropped: every universe firm keeps its row, and a
share is NaN when the firm is not matched to PARAT or its total
publications / patents are zero (a firm that publishes or patents nothing
has no defined AI share — which is not the same as a genuine 0 from a firm
that is active but not in AI). How to treat the NaNs is decided at
index-composition time.

Depends on the shared caches written by `structured_features_setup.ipynb`
for the same `UNIVERSE`. Writes
`data_clean/indicators/<UNIVERSE>/technology.parquet`.

In [ ]:
%load_ext autoreload
%autoreload 2

from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.indicators.common.io import load_cached_step, write_indicator
from src.indicators.structured_features import (
    INDICATOR,
    TECHNOLOGY_FEATURES,
    TECHNOLOGY_REASON_COLS,
    technology_indicator,
)

SHARED = INDICATOR
UNIVERSE = "sp500"

inputs = load_cached_step(SHARED, "inputs", UNIVERSE)
if inputs is None:
    raise RuntimeError(
        "Shared cache not found. Run notebooks/01_ingest_clean/"
        f"structured_features_setup.ipynb with UNIVERSE = {UNIVERSE!r} first."
    )
print(f"{len(inputs)} listed {UNIVERSE} firms in the input table")

## Compute the indicator (missing marked as NaN)

In [ ]:
indicator = technology_indicator(inputs)
_n_complete = int(inputs["technology_complete"].sum())
print(f"technology ({UNIVERSE}): {len(indicator)} firm rows, {_n_complete} with both shares defined")

_missing = inputs[~inputs["technology_complete"]]
_no_eto = _missing["eto_id"].isna()
print(f"missing {len(_missing)}: not in ETO PARAT {int(_no_eto.sum())}, "
      f"zero total publications {int(_missing['total_publications'].eq(0).sum())}, "
      f"zero total patents {int(_missing['total_patents'].eq(0).sum())} "
      f"(zero-denominator groups overlap)")
indicator.head()

## Sanity checks

Uniqueness of the join key, feature ranges (both shares are ratios of a
subset to its total, so [0, 1] holds by construction), the recomputed
shares against ETO's own rounded percentage columns, and the breakdown of
each `_reason` column -- confirming "unmatched" (no PARAT record) and
"zero_denominator" (matched, but zero/no total publications or patents)
line up with the counts above.

In [ ]:
assert indicator["normalized_company_name"].is_unique
for _col in TECHNOLOGY_FEATURES:
    assert indicator[_col].dropna().between(0, 1).all(), f"{_col} outside [0, 1]"

_kept = inputs[inputs["technology_complete"]]
_dev_pub = (_kept["ai_publication_share"] * 100 - _kept["eto_ai_publication_pct"]).abs().max()
_dev_pat = (_kept["ai_patent_share"] * 100 - _kept["eto_ai_patent_pct"]).abs().max()
print(f"cross-check vs ETO percentages: max deviation pub {_dev_pub:.3f}pp / patent {_dev_pat:.3f}pp")
assert _dev_pub < 0.15 and _dev_pat < 0.15

print("\n=== Feature distributions ===")
print(indicator[TECHNOLOGY_FEATURES].describe().to_string())

print("\n=== Missing-value reasons ===")
# Only the rows where the share is actually missing -- excluding those makes
# the breakdown read as "why is this one missing" instead of mixing in a
# confusing NaN count for the (majority) rows where nothing is wrong.
for _col in TECHNOLOGY_REASON_COLS:
    _reasons = indicator[_col].dropna()
    print(f"\n{_col}: {len(_reasons)} missing of {len(indicator)}")
    print(_reasons.value_counts().to_string() if len(_reasons) else "  (none)")

## Write the indicator parquet

Only the id columns plus the two ratio features: `compose_index` treats
every non-id column as a feature, so raw counts and match provenance stay
in the `structured_features` cache and the HTML review.

In [ ]:
path = write_indicator(indicator, "technology", UNIVERSE)
print(f"technology ({UNIVERSE}): {len(indicator)} firm rows -> {path}")
indicator.head()